# T1055 Process Injection NER Tagging

This notebook applies trained NER models to text files for extracting:
- **PROC**: Malicious processes
- **TARG**: Target libraries/processes
- **TTP**: TTP arguments

## Features:
- Smart sentence splitting that preserves filenames (service.exe) and paths (c:\\win\\serve.exe)
- Proper subword tokenization with BIO tag alignment
- Token-level prediction analysis
- Export to JSON/CSV formats

## 1. Setup and Imports

In [8]:
import torch
import re
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification
from typing import List, Dict, Tuple
from pathlib import Path

print("✓ Imports loaded")

✓ Imports loaded


## 2. Smart Sentence Splitting

This function preserves:
- File names: `service.exe`, `malware.dll`
- File paths: `c:\\windows\\system32\\cmd.exe`
- IP addresses: `192.168.1.1`
- Acronyms: `U.S.A.`, `D.L.L.`

In [9]:
def smart_sentence_split(text: str) -> List[str]:
    """
    Split text into sentences while preserving:
    - File names (e.g., service.exe, malware.dll)
    - File paths (e.g., c:\\windows\\system32\\cmd.exe)
    - IP addresses (e.g., 192.168.1.1)
    - Common abbreviations (e.g., Dr., Mr., etc.)
    
    Args:
        text: Input text to split
    
    Returns:
        List of sentences
    """
    # Replace problematic patterns with placeholders
    placeholders = {}
    counter = 0
    
    # Pattern 1: File paths (Windows and Unix)
    # Matches: c:\\windows\\system.exe, /usr/bin/bash, C:\\Users\\file.txt
    path_pattern = r'[a-zA-Z]:\\[\w\\.-]+|/[\w/.-]+'
    for match in re.finditer(path_pattern, text):
        placeholder = f"__FILEPATH_{counter}__"
        placeholders[placeholder] = match.group(0)
        text = text.replace(match.group(0), placeholder)
        counter += 1
    
    # Pattern 2: File names with extensions
    # Matches: service.exe, malware.dll, document.pdf
    filename_pattern = r'\b[\w-]+\.(exe|dll|sys|bat|cmd|vbs|ps1|jar|zip|rar|pdf|doc|docx|xls|xlsx)\b'
    for match in re.finditer(filename_pattern, text, re.IGNORECASE):
        placeholder = f"__FILENAME_{counter}__"
        placeholders[placeholder] = match.group(0)
        text = text.replace(match.group(0), placeholder)
        counter += 1
    
    # Pattern 3: IP addresses
    # Matches: 192.168.1.1, 10.0.0.1
    ip_pattern = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'
    for match in re.finditer(ip_pattern, text):
        placeholder = f"__IPADDR_{counter}__"
        placeholders[placeholder] = match.group(0)
        text = text.replace(match.group(0), placeholder)
        counter += 1
    
    # Pattern 4: Common abbreviations
    # Matches: Dr., Mr., Mrs., etc., i.e., e.g.
    abbrev_pattern = r'\b(Dr|Mr|Mrs|Ms|Prof|Sr|Jr|etc|i\.e|e\.g|vs)\.'
    for match in re.finditer(abbrev_pattern, text, re.IGNORECASE):
        placeholder = f"__ABBREV_{counter}__"
        placeholders[placeholder] = match.group(0)
        text = text.replace(match.group(0), placeholder)
        counter += 1
    
    # Now perform sentence splitting on cleaned text
    # Split on: . ! ? followed by whitespace and capital letter or end of string
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z]|$)', text)
    
    # Restore placeholders
    restored_sentences = []
    for sentence in sentences:
        for placeholder, original in placeholders.items():
            sentence = sentence.replace(placeholder, original)
        sentence = sentence.strip()
        if sentence:
            restored_sentences.append(sentence)
    
    return restored_sentences


# Test the function
test_text = """The malware service.exe injects into explorer.exe. It uses c:\\windows\\system32\\cmd.exe for execution. 
The C2 server is at 192.168.1.100. Dr. Smith analyzed the sample."""

test_sentences = smart_sentence_split(test_text)
print("Test sentence splitting:")
for i, sent in enumerate(test_sentences, 1):
    print(f"{i}. {sent}")

print("\n✓ Sentence splitting function loaded")

Test sentence splitting:
1. The malware service.exe injects into explorer.exe.
2. It uses c:\windows\system32\cmd.exe for execution.
3. The C2 server is at 192.168.1.100. Dr. Smith analyzed the sample.

✓ Sentence splitting function loaded


## 3. Subword Tokenization Functions

These functions handle RoBERTa tokenization and rejoin subwords back into complete words.

In [10]:
def rejoin_subwords(tokens: List[str]) -> List[str]:
    """
    Rejoin RoBERTa subword tokens back into complete words.
    RoBERTa uses 'Ġ' (\u0120) to indicate word boundaries.
    
    Args:
        tokens: List of subword tokens from tokenizer
    
    Returns:
        List of complete words
    """
    words = []
    current_word = ""
    
    for token in tokens:
        # Skip special tokens
        if token in ['<s>', '</s>', '<pad>', '<unk>']:
            continue
        
        # RoBERTa uses 'Ġ' (space character) to mark word boundaries
        if token.startswith('Ġ'):
            # Start of new word
            if current_word:
                words.append(current_word)
            current_word = token[1:]  # Remove the Ġ prefix
        else:
            # Continuation of current word
            current_word += token
    
    # Don't forget the last word
    if current_word:
        words.append(current_word)
    
    return words


def clean_text(text: str) -> str:
    """
    Clean text while preserving important cybersecurity artifacts.
    
    Args:
        text: Input text
    
    Returns:
        Cleaned text
    """
    # Replace tabs with spaces
    text = text.replace('\t', ' ')
    
    # Fix escaped quotes
    text = text.replace('\\\'', "'")
    
    # Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()


def split_long_sentence(text: str, tokenizer, max_length: int = 512) -> List[str]:
    """
    Split long sentences that exceed max_length into smaller chunks.
    
    Args:
        text: Input text
        tokenizer: HuggingFace tokenizer
        max_length: Maximum token length
    
    Returns:
        List of text chunks
    """
    # Tokenize to check length
    tokens = tokenizer.tokenize(text)
    
    # If within limit, return as-is
    if len(tokens) <= max_length:
        return [text]
    
    # Split by punctuation or conjunctions
    chunks = re.split(r'[,;]\s+|\s+(?:and|or|but)\s+', text)
    
    result = []
    current_chunk = ""
    
    for chunk in chunks:
        test_chunk = current_chunk + " " + chunk if current_chunk else chunk
        test_tokens = tokenizer.tokenize(test_chunk)
        
        if len(test_tokens) <= max_length:
            current_chunk = test_chunk
        else:
            if current_chunk:
                result.append(current_chunk.strip())
            current_chunk = chunk
    
    if current_chunk:
        result.append(current_chunk.strip())
    
    return result if result else [text]

print("✓ Tokenization functions loaded")

✓ Tokenization functions loaded


## 4. NER Model Class

Loads the trained model and handles prediction with proper BIO tag alignment.

In [11]:
class T1055NERTagger:
    """
    T1055 Process Injection NER tagger with proper tokenization alignment.
    """
    
    def __init__(self, model_checkpoint: str):
        """
        Initialize the NER tagger.
        
        Args:
            model_checkpoint: Path to trained model
        """
        print(f"Loading T1055 NER model from: {model_checkpoint}")
        
        # Load tokenizer with add_prefix_space for RoBERTa
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_checkpoint, 
            add_prefix_space=True
        )
        
        # Load model
        self.model = AutoModelForTokenClassification.from_pretrained(
            model_checkpoint
        )
        
        # Get label mappings
        self.id2label = self.model.config.id2label
        self.label2id = self.model.config.label2id
        self.label_names = [self.id2label[i] for i in range(len(self.id2label))]
        
        # Move to GPU if available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        
        print(f"✓ Model loaded")
        print(f"  Device: {self.device}")
        print(f"  Labels: {self.label_names}")
    
    def align_labels_with_tokens(self, labels: List[int], word_ids: List[int]) -> List[int]:
        """
        Align predictions with tokens, converting B- to I- for subwords.
        
        Args:
            labels: List of label IDs
            word_ids: Word IDs from tokenizer
        
        Returns:
            Aligned label list
        """
        new_labels = []
        current_word = None
        
        for word_id in word_ids:
            if word_id != current_word:
                # Start of a new word
                current_word = word_id
                label = -100 if word_id is None else labels[word_id]
                new_labels.append(label)
            
            elif word_id is None:
                # Special token
                new_labels.append(-100)
            
            else:
                # Subword continuation - convert B- to I-
                label = labels[word_id]
                label_name = self.id2label[label]
                
                if label_name.startswith('B-'):
                    entity_type = label_name[2:]
                    i_label_name = f'I-{entity_type}'
                    label = self.label2id[i_label_name]
                
                new_labels.append(label)
        
        return new_labels
    
    def predict_sentence(self, sentence: str) -> Dict:
        """
        Predict entities in a sentence.
        
        Args:
            sentence: Input sentence
        
        Returns:
            Dict with tokens, predictions, and entities
        """
        # Split into tokens (whitespace)
        tokens = sentence.split()
        
        if not tokens:
            return {
                'sentence': sentence,
                'tokens': [],
                'predictions': [],
                'entities': []
            }
        
        # Tokenize
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        
        # Get word_ids BEFORE moving to device
        word_ids = encoding.word_ids()
        
        # Move to device
        encoding_tensors = {k: v.to(self.device) for k, v in encoding.items()}
        
        # Predict
        with torch.no_grad():
            outputs = self.model(**encoding_tensors)
            logits = outputs.logits[0].cpu()
            predictions = logits.argmax(-1).tolist()
            probabilities = torch.softmax(logits, dim=-1)
        
        # Align predictions with words (first subword only)
        word_predictions = {}
        word_confidences = {}
        current_word = None
        
        for token_idx, word_id in enumerate(word_ids):
            if word_id is None:
                continue
            
            # Only take first subword prediction
            if word_id != current_word:
                current_word = word_id
                pred_id = predictions[token_idx]
                pred_label = self.label_names[pred_id]
                confidence = probabilities[token_idx][pred_id].item()
                word_predictions[word_id] = pred_label
                word_confidences[word_id] = confidence
        
        # Create token-level results
        token_results = []
        for word_id in sorted(word_predictions.keys()):
            token_results.append({
                'token': tokens[word_id],
                'prediction': word_predictions[word_id],
                'confidence': word_confidences[word_id]
            })
        
        # Extract entities
        entities = self.extract_entities_from_predictions(tokens, word_predictions)
        
        return {
            'sentence': sentence,
            'tokens': token_results,
            'entities': entities,
            'n_entities': len(entities)
        }
    
    def extract_entities_from_predictions(
        self, 
        tokens: List[str], 
        predictions: Dict[int, str]
    ) -> List[Dict]:
        """
        Extract entities from token predictions.
        
        Args:
            tokens: List of tokens
            predictions: Dict mapping word_id to prediction label
        
        Returns:
            List of entity dicts
        """
        entities = []
        current_entity = None
        current_tokens = []
        
        for word_idx in sorted(predictions.keys()):
            label = predictions[word_idx]
            token = tokens[word_idx]
            
            if label.startswith('B-'):
                # Start of new entity
                if current_entity:
                    entities.append({
                        'text': ' '.join(current_tokens),
                        'label': current_entity,
                        'tokens': current_tokens.copy()
                    })
                
                current_entity = label[2:]
                current_tokens = [token]
            
            elif label.startswith('I-') and current_entity:
                # Continuation
                entity_type = label[2:]
                if entity_type == current_entity:
                    current_tokens.append(token)
                else:
                    # Entity type changed
                    entities.append({
                        'text': ' '.join(current_tokens),
                        'label': current_entity,
                        'tokens': current_tokens.copy()
                    })
                    current_entity = entity_type
                    current_tokens = [token]
            
            else:
                # O tag
                if current_entity:
                    entities.append({
                        'text': ' '.join(current_tokens),
                        'label': current_entity,
                        'tokens': current_tokens.copy()
                    })
                    current_entity = None
                    current_tokens = []
        
        # Don't forget last entity
        if current_entity:
            entities.append({
                'text': ' '.join(current_tokens),
                'label': current_entity,
                'tokens': current_tokens.copy()
            })
        
        return entities

print("✓ NER Tagger class loaded")

✓ NER Tagger class loaded


## 5. Main Processing Function

In [12]:
def process_text_file(
    input_path: str,
    model_path: str,
    output_json: str = None,
    output_csv: str = None,
    verbose: bool = True
) -> Dict:
    """
    Process a text file with NER tagging.
    
    Args:
        input_path: Path to input .txt file
        model_path: Path to trained NER model
        output_json: Optional path to save JSON results
        output_csv: Optional path to save CSV results
        verbose: Print progress
    
    Returns:
        Dict with all results
    """
    if verbose:
        print("\n" + "="*80)
        print("T1055 NER TAGGING")
        print("="*80)
        print(f"Input: {input_path}")
        print(f"Model: {model_path}")
    
    # Read file
    with open(input_path, 'r', encoding='utf-8') as f:
        text = f.read()
    
    if verbose:
        print(f"✓ Loaded {len(text)} characters")
    
    # Clean text
    text = clean_text(text)
    
    # Split into sentences
    sentences = smart_sentence_split(text)
    
    if verbose:
        print(f"✓ Split into {len(sentences)} sentences")
    
    # Load model
    tagger = T1055NERTagger(model_path)
    
    # Process sentences
    all_results = []
    total_entities = 0
    sentences_with_entities = 0
    
    if verbose:
        print(f"\n🔍 Processing sentences...")
    
    for i, sentence in enumerate(sentences):
        result = tagger.predict_sentence(sentence)
        result['sentence_id'] = i
        all_results.append(result)
        
        if result['n_entities'] > 0:
            total_entities += result['n_entities']
            sentences_with_entities += 1
            
            if verbose:
                print(f"  [{i+1}/{len(sentences)}] {result['n_entities']} entities: {sentence[:60]}...")
    
    # Compile results
    results = {
        'input_file': input_path,
        'model_path': model_path,
        'total_sentences': len(sentences),
        'sentences_with_entities': sentences_with_entities,
        'total_entities': total_entities,
        'sentences': all_results
    }
    
    # Save JSON
    if output_json:
        with open(output_json, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        if verbose:
            print(f"\n✓ Saved JSON to: {output_json}")
    
    # Save CSV
    if output_csv:
        export_to_csv(results, output_csv, verbose)
    
    if verbose:
        print("\n" + "="*80)
        print("SUMMARY")
        print("="*80)
        print(f"Total sentences: {len(sentences)}")
        print(f"Sentences with entities: {sentences_with_entities}")
        print(f"Total entities extracted: {total_entities}")
        print("="*80 + "\n")
    
    return results


def export_to_csv(results: Dict, output_path: str, verbose: bool = True):
    """
    Export results to CSV format.
    
    Args:
        results: Results dict from process_text_file
        output_path: Path to save CSV
        verbose: Print progress
    """
    rows = []
    
    for sent_result in results['sentences']:
        sent_id = sent_result['sentence_id']
        sentence = sent_result['sentence']
        
        if sent_result['entities']:
            for entity in sent_result['entities']:
                rows.append({
                    'sentence_id': sent_id,
                    'sentence': sentence,
                    'entity_text': entity['text'],
                    'entity_label': entity['label']
                })
    
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(output_path, index=False)
        if verbose:
            print(f"✓ Saved CSV to: {output_path} ({len(rows)} rows)")
    else:
        if verbose:
            print("⚠️  No entities to export")


def display_results(results: Dict, max_sentences: int = 10):
    """
    Display results in formatted output.
    
    Args:
        results: Results dict from process_text_file
        max_sentences: Maximum sentences to display
    """
    print("\n" + "="*90)
    print("EXTRACTED ENTITIES")
    print("="*90)
    
    displayed = 0
    for sent_result in results['sentences']:
        if sent_result['n_entities'] == 0:
            continue
        
        if displayed >= max_sentences:
            remaining = sum(1 for s in results['sentences'] if s['n_entities'] > 0) - displayed
            if remaining > 0:
                print(f"\n... and {remaining} more sentences with entities")
            break
        
        print(f"\n{'='*90}")
        print(f"SENTENCE #{sent_result['sentence_id'] + 1}")
        print(f"{'='*90}")
        print(f"{sent_result['sentence']}")
        print(f"\nEntities ({sent_result['n_entities']}):")
        
        for entity in sent_result['entities']:
            if entity['label'] == 'PROC':
                print(f"  🔴 PROC: {entity['text']}")
            elif entity['label'] == 'TARG':
                print(f"  🎯 TARG: {entity['text']}")
            elif entity['label'] == 'TTP':
                print(f"  ⚙️  TTP: {entity['text']}")
        
        displayed += 1
    
    print("\n" + "="*90 + "\n")

print("✓ Processing functions loaded")

✓ Processing functions loaded


## 6. Run NER Tagging

**Configure paths and run:**

In [13]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Model path
ner_model_path = "/home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480"

# Input file
input_file = "/home/jovyan/work/SampleReports/StuxNet.txt"

# Output files (optional)
output_json = "/home/jovyan/work/Results/stuxnet_ner_results.json"
output_csv = "/home/jovyan/work/Results/stuxnet_ner_entities.csv"

# ============================================================================
# RUN PROCESSING
# ============================================================================

results = process_text_file(
    input_path=input_file,
    model_path=ner_model_path,
    output_json=output_json,
    output_csv=output_csv,
    verbose=True
)

# Display results
display_results(results, max_sentences=10)


T1055 NER TAGGING
Input: /home/jovyan/work/SampleReports/StuxNet.txt
Model: /home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480
✓ Loaded 24513 characters
✓ Split into 155 sentences
Loading T1055 NER model from: /home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480


The tokenizer you are loading from '/home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✓ Model loaded
  Device: cuda
  Labels: ['B-MalProcess', 'B-TargetProcess', 'B-MalLibrary', 'B-TargetLibrary', 'B-TTPArgument', 'I-MalProcess', 'I-TargetProcess', 'I-MalLibrary', 'I-TargetLibrary', 'I-TTPArgument', 'O']

🔍 Processing sentences...
  [77/155] 1 entities: Export 16 is then called to continue installation: it inject...
  [80/155] 2 entities: If all these checks pass, it will drop two driver files to i...
  [81/155] 2 entities: It is unclear if there is any special significance to the cu...
  [87/155] 1 entities: Step7 programs, which control and monitor these PLCs, use a ...
  [101/155] 1 entities: By intercepting calls to s7otbxdx.dll, Stuxnet hides the mal...

✓ Saved JSON to: /home/jovyan/work/Results/stuxnet_ner_results.json
✓ Saved CSV to: /home/jovyan/work/Results/stuxnet_ner_entities.csv (7 rows)

SUMMARY
Total sentences: 155
Sentences with entities: 5
Total entities extracted: 7


EXTRACTED ENTITIES

SENTENCE #77
Export 16 is then called to continue installation: i

## 7. Analyze Token-Level Predictions (Debug)

If results are poor, analyze token-level predictions:

In [7]:
def analyze_token_predictions(sentence: str, tagger: T1055NERTagger):
    """
    Detailed analysis of token-level predictions.
    """
    print("\n" + "="*90)
    print("TOKEN PREDICTION ANALYSIS")
    print("="*90)
    print(f"Sentence: {sentence}\n")
    
    result = tagger.predict_sentence(sentence)
    
    print(f"{'Token':<30} {'Prediction':<15} {'Confidence':<10}")
    print("-"*90)
    
    for token_info in result['tokens']:
        token = token_info['token']
        pred = token_info['prediction']
        conf = token_info['confidence']
        
        marker = "⭐" if pred != 'O' else "  "
        print(f"{marker} {token:<28} {pred:<15} {conf:.3f}")
    
    print("\n" + "="*90 + "\n")

# Load tagger
tagger = T1055NERTagger(ner_model_path)

# Test specific sentence
test_sentence = "The malware service.exe injects malicious code into explorer.exe using CreateRemoteThread"
analyze_token_predictions(test_sentence, tagger)

Loading T1055 NER model from: /home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480


The tokenizer you are loading from '/home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✓ Model loaded
  Device: cuda
  Labels: ['B-MalProcess', 'B-TargetProcess', 'B-MalLibrary', 'B-TargetLibrary', 'B-TTPArgument', 'I-MalProcess', 'I-TargetProcess', 'I-MalLibrary', 'I-TargetLibrary', 'I-TTPArgument', 'O']

TOKEN PREDICTION ANALYSIS
Sentence: The malware service.exe injects malicious code into explorer.exe using CreateRemoteThread

Token                          Prediction      Confidence
------------------------------------------------------------------------------------------
   The                          O               1.000
   malware                      O               1.000
⭐ service.exe                  B-MalProcess    1.000
   injects                      O               1.000
   malicious                    O               1.000
   code                         O               1.000
   into                         O               1.000
⭐ explorer.exe                 B-TargetProcess 1.000
   using                        O               1.000
   CreateRemoteThre

## 8. Export All Token Predictions

Export every token and its prediction for detailed analysis:

In [14]:
def export_all_token_predictions(
    input_path: str,
    model_path: str,
    output_path: str
):
    """
    Export all token predictions to CSV.
    """
    print(f"\n🔍 Exporting all token predictions...")
    
    # Read and process
    with open(input_path, 'r', encoding='utf-8') as f:
        text = f.read()
    
    text = clean_text(text)
    sentences = smart_sentence_split(text)
    
    # Load tagger
    tagger = T1055NERTagger(model_path)
    
    # Process all
    all_tokens = []
    
    for i, sentence in enumerate(sentences):
        result = tagger.predict_sentence(sentence)
        
        for token_info in result['tokens']:
            all_tokens.append({
                'sentence_id': i,
                'sentence': sentence,
                'token': token_info['token'],
                'prediction': token_info['prediction'],
                'confidence': token_info['confidence']
            })
    
    # Export
    df = pd.DataFrame(all_tokens)
    df.to_csv(output_path, index=False)
    
    print(f"✓ Exported {len(all_tokens)} token predictions to: {output_path}")
    print(f"\n📊 Prediction counts:")
    print(df['prediction'].value_counts())
    
    return df

# Export all tokens
df_tokens = export_all_token_predictions(
    input_file,
    ner_model_path,
    "/home/jovyan/work/Results/all_token_predictions.csv"
)


🔍 Exporting all token predictions...
Loading T1055 NER model from: /home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480


The tokenizer you are loading from '/home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✓ Model loaded
  Device: cuda
  Labels: ['B-MalProcess', 'B-TargetProcess', 'B-MalLibrary', 'B-TargetLibrary', 'B-TTPArgument', 'I-MalProcess', 'I-TargetProcess', 'I-MalLibrary', 'I-TargetLibrary', 'I-TTPArgument', 'O']
✓ Exported 3655 token predictions to: /home/jovyan/work/Results/all_token_predictions.csv

📊 Prediction counts:
prediction
O                  3648
B-TargetProcess       3
B-MalLibrary          2
B-TargetLibrary       2
Name: count, dtype: int64


## 9. Quick Statistics

In [ ]:
# Get statistics from results
if 'results' in locals():
    print("\n📊 STATISTICS")
    print("="*60)
    print(f"Total sentences: {results['total_sentences']}")
    print(f"Sentences with entities: {results['sentences_with_entities']}")
    print(f"Total entities: {results['total_entities']}")
    
    # Entity type breakdown
    entity_counts = {'PROC': 0, 'TARG': 0, 'TTP': 0}
    for sent in results['sentences']:
        for ent in sent['entities']:
            entity_counts[ent['label']] += 1
    
    print(f"\nEntity breakdown:")
    print(f"  PROC (Malicious Processes): {entity_counts['PROC']}")
    print(f"  TARG (Target Libraries/Processes): {entity_counts['TARG']}")
    print(f"  TTP (TTP Arguments): {entity_counts['TTP']}")
    print("="*60)
else:
    print("⚠️  Run the processing cell first to generate results")